In [1]:
import optuna

from optuna.integration import XGBoostPruningCallback

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

/Users/cinar/.pyenv/versions/3.13.14/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [3]:
def objective(trial):
    max_depth=trial.suggest_int("max_depth",3,8)
    learning_rate=trial.suggest_float("learning_rate",0.01,0.3)
    n_estimators=trial.suggest_int("n_estimators",50,300)

    model=XGBClassifier(max_depth=max_depth,learning_rate=learning_rate,n_estimators=n_estimators,
                        eval_metric="logloss",random_state=1)

    pruning_callback=XGBoostPruningCallback(trial,"validation_0-logloss")

    model.fit(X_train,y_train,eval_set=[(X_test,y_test)],callbacks=[pruning_callback],verbose=False)

    predictions=model.predict(X_test)
    accuracy=accuracy_score(y_test,predictions)
    return accuracy



In [4]:
study=optuna.create_study(direction="maximize",pruner=optuna.pruners.MedianPruner())

[I 2026-07-06 21:47:38,914] A new study created in memory with name: no-name-4d56d00e-22c5-4cca-8a8f-f4edd8a7eba5


In [7]:
study.optimize(objective,n_trials=30)

/Users/cinar/.pyenv/versions/3.13.14/lib/python3.13/site-packages/xgboost/sklearn.py:835: UserWarning: `callbacks` in `fit` method is deprecated for better compatibility with scikit-learn, use `callbacks` in constructor or`set_params` instead.
  warnings.warn(
[I 2026-07-06 21:51:16,581] Trial 30 pruned. Trial was pruned at iteration 0.
[I 2026-07-06 21:51:16,745] Trial 31 finished with value: 0.89 and parameters: {'max_depth': 3, 'learning_rate': 0.034599392895856174, 'n_estimators': 272}. Best is trial 26 with value: 0.91.
/Users/cinar/.pyenv/versions/3.13.14/lib/python3.13/site-packages/xgboost/sklearn.py:835: UserWarning: `callbacks` in `fit` method is deprecated for better compatibility with scikit-learn, use `callbacks` in constructor or`set_params` instead.
  warnings.warn(
[I 2026-07-06 21:51:16,870] Trial 32 finished with value: 0.895 and parameters: {'max_depth': 3, 'learning_rate': 0.06531488947817313, 'n_estimators': 283}. Best is trial 26 with value: 0.91.
/Users/cinar/.py

In [9]:
len(study.trials)

60

In [10]:
pruned=len([

    t for t in study.trials 
    if t.state == optuna.trial.TrialState.PRUNED

])

In [11]:
pruned 

14

In [12]:
study.best_value

0.91